# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [ ]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [ ]:
def ajouter_features(df):
    """Ajoute un indicateur simple de complétude, de manière reproductible."""
    df=df.copy()
    colonnes_num=[c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if 'nb_champs_numeriques_renseignes' not in df.columns:
        df['nb_champs_numeriques_renseignes']=df[colonnes_num].notna().sum(axis=1) if colonnes_num else 0
    return df

### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")



### Le tableau d'essais


In [ ]:
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [ ]:
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [ ]:

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique() if 'zone_clinique' in train.columns else 0} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

IDENTIFIANT_GITHUB = "REMPLACER_PAR_VOTRE_IDENTIFIANT_GITHUB"          
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != 'REMPLACER_PAR_VOTRE_IDENTIFIANT_GITHUB':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail avec la feature, et actualise les découpages."""
    global modele, X, Xtr, Xval, ytr, yval, X_num, Xtr_n, Xval_n, modele_lineaire
    X = ajouter_features(X_sans_features)
    Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
    X_num = pd.get_dummies(X, columns=CAT).fillna(0)
    Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]
    modele_lineaire = LogisticRegression(solver="newton-cholesky", max_iter=500).fit(Xtr_n, ytr)
    return 100 * modele.score(Xval, yval)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

n_colonnes_avant = X.shape[1]

AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')



### L'interface, telle qu'on l'appelle


In [ ]:
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt      

## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [ ]:
# Q1
part_absents_controle = 100 * float(yval.mean())
ma_reference = 100 * max(float(yval.mean()), 1-float(yval.mean()))
pred = np.full(len(yval), int(float(ytr.mean()) >= 0.5), dtype=int)
ecart_a_train = ma_reference - reference
barre_q1 = max(float(erreur_type), 1e-9)
ecart_en_barres = (83-ma_reference)/barre_q1
reponse_au_collegue = (f"La référence sur le contrôle vaut {ma_reference:.2f} %. Un score annoncé de 83 % est donc à {83-ma_reference:+.2f} points de la référence, soit {ecart_en_barres:+.2f} barre(s) de bruit. Sans dépasser la référence de deux barres, ce n'est pas une amélioration convaincante.")
print(f"Absents={part_absents_controle:.2f} %, référence contrôle={ma_reference:.2f} %, référence train={reference:.2f} %")

### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [ ]:
# Q2 : balayage de toutes les colonnes puis filtre des groupes de 500 lignes

def balayage_colonnes(df):
    rows=[]
    for col in df.columns:
        if col == CIBLE: continue
        g=df.groupby(col,dropna=False)[CIBLE].agg(['mean','size'])
        if len(g): rows.append({'colonne':col,'ecart_points':100*(g['mean'].max()-g['mean'].min()),'groupes':len(g),'groupes_500':int((g['size']>=500).sum())})
    return pd.DataFrame(rows).set_index('colonne').sort_values('ecart_points',ascending=False)

balayage_brut=balayage_colonnes(mon_train)
colonne_piegee=str(balayage_brut.index[0])
ecart_non_filtre=float(balayage_brut.iloc[0]['ecart_points'])
groupes_de_500=int(balayage_brut.iloc[0]['groupes_500'])
rows=[]
for col in mon_train.columns:
    if col==CIBLE: continue
    g=mon_train.groupby(col,dropna=False)[CIBLE].agg(['mean','size'])
    g=g[g['size']>=500]
    if len(g)>=2: rows.append({'colonne':col,'ecart_points':100*(g['mean'].max()-g['mean'].min()),'plus_petit_groupe':int(g['size'].min()),'groupes_retenus':len(g)})
classement=pd.DataFrame(rows).set_index('colonne').sort_values('ecart_points',ascending=False) if rows else pd.DataFrame(columns=['ecart_points','plus_petit_groupe','groupes_retenus'])
if len(classement):
    plus_petit_groupe_du_classement=int(classement.iloc[0]['plus_petit_groupe']); meilleure_colonne=str(classement.index[0]); ecart_filtre=float(classement.iloc[0]['ecart_points'])
else:
    plus_petit_groupe_du_classement=0; meilleure_colonne='aucune'; ecart_filtre=0.0
ce_que_le_critere_ne_protege_pas=f"Le seuil de 500 lignes écarte les modalités trop rares, mais ne garantit ni que la colonne soit connue au moment de décider ni qu'elle ne révèle pas indirectement la cible. La colonne {meilleure_colonne} doit encore être vérifiée selon son sens métier et son moment de disponibilité."
lecture=f"Sans filtre, {colonne_piegee} gagne avec {ecart_non_filtre:.2f} points, mais seulement {groupes_de_500} groupe(s) atteignent 500 lignes : un écart extrême peut venir d'une valeur rare. Après filtrage, {meilleure_colonne} arrive en tête avec {ecart_filtre:.2f} points."
display(balayage_brut.head(10)); display(classement.head(10))

### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [ ]:
# Q3 : deux constats supplémentaires mesurés sur les données
nb_lignes=len(train); nb_doublons=int(train.duplicated().sum())
constat("des lignes dupliquées sont présentes ou absentes", f"{nb_doublons} doublon(s) sur {nb_lignes} lignes", f"contre 0 doublon attendu sur {nb_lignes} lignes distinctes", "écarter les doublons s'ils sont accidentels, sinon mesurer leur origine")
colonnes_constantes=[c for c in train.columns if c!=CIBLE and train[c].nunique(dropna=False)<=1]
constat("colonnes constantes détectées", f"{len(colonnes_constantes)} colonne(s) constante(s)", f"sur {len(train.columns)-1} colonnes descriptives examinées", "écarter les colonnes constantes" if colonnes_constantes else "surveiller : aucune colonne constante détectée")

### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [ ]:
# Q4 : prédictions annoncées avant les mesures
pred_age=-0.5; pred_doublons=0.0; pred_taille=0.0
barre_q4=max(float(erreur_type),1e-9)
def score_avec_sans_colonne(col):
    if col not in X.columns: return float(score_interne),float(score_interne)
    ma=HistGradientBoostingClassifier(random_state=0).fit(Xtr,ytr)
    sa=100*ma.score(Xval,yval)
    mb=HistGradientBoostingClassifier(random_state=0).fit(Xtr.drop(columns=[col]),ytr)
    sb=100*mb.score(Xval.drop(columns=[col]),yval)
    return sa,sb
s_avec_age,s_sans_age=score_avec_sans_colonne('age')
age_en_barres=(s_avec_age-s_sans_age)/barre_q4
if 'age' in X_num.columns:
    cols_lin=[c for c in X_num.columns if c!='age']
    ml_a=LogisticRegression(solver='newton-cholesky',max_iter=500).fit(Xtr_n,ytr)
    ml_b=LogisticRegression(solver='newton-cholesky',max_iter=500).fit(Xtr_n[cols_lin],ytr)
    lin_avec_age=100*ml_a.score(Xval_n,yval); lin_sans_age=100*ml_b.score(Xval_n[cols_lin],yval)
else: lin_avec_age=float(score_lineaire); lin_sans_age=float(score_lineaire)
lin_age_en_barres=(lin_avec_age-lin_sans_age)/barre_q4
verdict_age=f"Retirer age change le score des arbres de {s_sans_age-s_avec_age:+.2f} points et celui du linéaire de {lin_sans_age-lin_avec_age:+.2f} points. Un arbre peut isoler une valeur impossible par une coupure, tandis qu'un modèle linéaire peut lui attribuer un poids numérique ; le verdict dépend des deux mesures et du sens métier."
verdict_doublons=f"Le contrôle trouve {nb_doublons} doublon(s) exact(s) sur {len(train)} lignes. Si le résultat est nul, cela signifie seulement qu'aucun doublon exact n'a été détecté, pas que toutes les données sont correctes."

### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [ ]:
# Q5 : vingt sous-échantillons pour chaque taille
rng=np.random.default_rng(2026)
def scores_sous_echantillons(n,repetitions=20):
    n_eff=min(int(n),len(Xval)); scores=[]
    for _ in range(repetitions):
        idx=rng.choice(len(Xval),size=n_eff,replace=False)
        scores.append(100*float(modele.score(Xval.iloc[idx],yval.iloc[idx])))
    return scores
ecarts_1000=scores_sous_echantillons(1000); ecarts_5000=scores_sous_echantillons(5000)
ecart_type_1000=float(np.std(ecarts_1000,ddof=1)); ecart_type_5000=float(np.std(ecarts_5000,ddof=1))
p=float(yval.mean())
formule_1000=100*np.sqrt(p*(1-p)/min(1000,len(Xval))); formule_5000=100*np.sqrt(p*(1-p)/min(5000,len(Xval)))
rapport_1000=ecart_type_1000/max(formule_1000,1e-9); rapport_5000=ecart_type_5000/max(formule_5000,1e-9)
lecture=f"La formule prévoit une barre plus petite à 5 000 qu'à 1 000 lignes (environ un facteur √5 si les deux tailles sont disponibles). Les rapports observé/formule sont {rapport_1000:.2f} et {rapport_5000:.2f}. Vingt tirages donnent une estimation encore incertaine de l'écart-type, et les tirages partagent la même validation."

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
# Q6 : tirer 25 000 lignes si la validation le permet, sinon utiliser toute la validation
n_25000=min(25000,len(Xval)); ecarts_25000=scores_sous_echantillons(n_25000)
ecart_type_25000=float(np.std(ecarts_25000,ddof=1))
formule_25000=100*np.sqrt(float(yval.mean())*(1-float(yval.mean()))/n_25000)
rapport_25000=ecart_type_25000/max(formule_25000,1e-9)
correction_population_finie=np.sqrt(max(len(Xval)-n_25000,0)/max(len(Xval)-1,1))
rapport_corrige_25000=rapport_25000/correction_population_finie if correction_population_finie>0 else float('inf')
pourquoi_la_formule_deborde=f"La formule simple suppose des observations indépendantes d'une population théorique ; ici, le tirage est sans remise dans {len(Xval)} lignes. On a pu tirer {n_25000} lignes et la correction de population finie vaut {correction_population_finie:.4f}. Les rapports pour 1 000, 5 000 et ce tirage sont {rapport_1000:.2f}, {rapport_5000:.2f} et {rapport_25000:.2f} : la variation diminue quand on approche de toute la validation."

### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
# Q7 : seul learning_rate change
pred_pas="Je prévois que 0,05 apprend plus lentement et peut nécessiter davantage d'itérations, 0,5 sera intermédiaire, et 2 peut être instable ou s'arrêter tôt. L'arrêt anticipé peut empêcher les scores de suivre simplement le pas."
rows=[]
for lr in [0.05,0.5,2.0]:
    m=HistGradientBoostingClassifier(random_state=0,learning_rate=lr).fit(Xtr,ytr); sc=100*m.score(Xval,yval)
    rows.append({'learning_rate':lr,'score_validation':sc,'arbres_construits':int(m.n_iter_),'ecart_reference_en_barres':(sc-reference)/max(float(erreur_type),1e-9)})
reglages=pd.DataFrame(rows)
score_separe=f"Les scores varient de {reglages.score_validation.min():.2f} % à {reglages.score_validation.max():.2f} %, soit {(reglages.score_validation.max()-reglages.score_validation.min())/max(float(erreur_type),1e-9):.2f} barre(s) de bruit ; la séparation dépasse deux barres : {(reglages.score_validation.max()-reglages.score_validation.min())>2*erreur_type}."
ce_qui_borne=f"Le nombre d'itérations construites varie de {reglages.arbres_construits.min()} à {reglages.arbres_construits.max()}. L'arrêt anticipé borne donc aussi l'apprentissage ; comparez ces valeurs à la prédiction initiale avant de conclure que le plus grand pas est meilleur."
display(reglages)

### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
# Q8 : trois essais de natures différentes
barre_q8=max(float(erreur_type),1e-9); base_score=float(score_interne)
col_candidat=next((c for c in X.columns if c not in A_ECARTER and pd.api.types.is_numeric_dtype(X[c]) and c!=CIBLE),None)
prevoir("retirer une colonne numérique","variation faible, probablement dans le bruit","si la colonne est redondante, son retrait devrait peu changer le score")
def mesure_sans_colonne():
    if col_candidat is None: return base_score/100
    m=HistGradientBoostingClassifier(random_state=0).fit(Xtr.drop(columns=[col_candidat]),ytr)
    return m.score(Xval.drop(columns=[col_candidat]),yval)
s1=essai("retirer une colonne numérique",mesure_sans_colonne,f"retrait de {col_candidat}","variation faible attendue")
prevoir("learning_rate 0.05","score proche du modèle de base","seul le pas d'apprentissage change")
def mesure_lr(): return HistGradientBoostingClassifier(random_state=0,learning_rate=.05).fit(Xtr,ytr).score(Xval,yval)
s2=essai("learning_rate 0.05",mesure_lr,"learning_rate=0.05","score proche attendu")
prevoir("régression logistique","score éventuellement inférieur aux arbres","modèle linéaire moins flexible")
s3=essai("régression logistique",lambda:modele_lineaire.score(Xval_n,yval),"LogisticRegression","comparaison de famille")
def ligne_essai(nom,nature,attendu,score):
    score=base_score if score is None else float(score)
    gain=(score-base_score)/barre_q8
    return {'essai':nom,'nature':nature,'attendu':attendu,'mesuré':score,'gain_en_barres':gain,'verdict':'réel' if abs(gain)>2 else 'dans le bruit'}
mes_essais=pd.DataFrame([ligne_essai('retirer une colonne numérique','donnée','variation faible',s1),ligne_essai('learning_rate 0.05','réglage','score proche',s2),ligne_essai('régression logistique','famille de modèle','potentiellement inférieur',s3 if s3 is not None else score_lineaire)])
optimisme="Avant le second tirage, je m'attends à ce que le signe du gain soit similaire, mais son amplitude puisse changer ; un gain qui disparaît ou change de signe était probablement du bruit."
rng2=np.random.default_rng(42); idx2=rng2.choice(len(X),size=max(2,int(.8*len(X))),replace=False); val2=np.setdiff1d(np.arange(len(X)),idx2)
if len(val2):
    mt=HistGradientBoostingClassifier(random_state=17,learning_rate=.05).fit(X.iloc[idx2],y.iloc[idx2]); mb=HistGradientBoostingClassifier(random_state=17).fit(X.iloc[idx2],y.iloc[idx2])
    verifie_par_second_tirage=100*(mt.score(X.iloc[val2],y.iloc[val2])-mb.score(X.iloc[val2],y.iloc[val2]))/barre_q8
else: verifie_par_second_tirage=0.0
display(mes_essais)

### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
# Q9 : différence des scores exprimée en barres et lecture prudente des coefficients
rapport=(score_interne-score_lineaire)/max(float(erreur_type),1e-9)
num_cols=[c for c in X.columns if pd.api.types.is_numeric_dtype(X[c]) and c not in {'age','age_mois','montant_acte','cout_acte','dossier_id','patient_id'} and X[c].nunique(dropna=True)>1]
cat_cols=[c for c in X.columns if c not in num_cols and c not in {'age','age_mois','montant_acte','cout_acte'}]
trois_coefficients=(num_cols+cat_cols)[:3]
pourquoi_pas_les_dupliquees="age et age_mois, ainsi que montant_acte et cout_acte, représentent presque la même information ; les garder ensemble rend l'attribution du coefficient instable et n'isole pas l'effet propre de chaque variable."
quartier_provoque="Non : un coefficient décrit une association conditionnelle, pas une cause. Le quartier peut représenter des facteurs confondants comme la distance, l'accès ou l'organisation ; une analyse causale serait nécessaire."
cout_et_rapport="La régression logistique coûte la préparation des catégories en indicatrices, le traitement des valeurs manquantes et idéalement la mise à l'échelle. En échange, elle fournit des coefficients signés et lisibles comme associations, mais pas automatiquement causales."

### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
# Q10 : mesurer la feature et contourner volontairement preparer()
constat_d_origine="colonnes constantes détectées"
prevoir("nb_champs_numeriques_renseignes","gain faible ou nul plausible","la complétude de ligne peut résumer l'information disponible")
score_feature=remesurer()
gain_de_ma_feature=(score_feature-score_interne)/max(float(erreur_type),1e-9)
# On oublie ajouter_features et on remplace la feature manquante par zéro : démonstration du contournement.
Xval_nue=X_sans_features.loc[Xval.index].copy()
Xval_contournee=Xval_nue.reindex(columns=X.columns,fill_value=0)
score_contourne=100*float(modele.score(Xval_contournee,yval))
ecart_du_contournement=(score_contourne-score_feature)/max(float(erreur_type),1e-9)
quelle_etape_oubliee="L_etape ajouter_features() de preparer() a ete oubliee : le modèle reçoit zéro à la place de la complétude réelle de la ligne."

### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
# Q11 : écart de remplissage entre train et test
cols_communes=[c for c in train.columns if c in test.columns and c!=CIBLE]
rempl_train=train[cols_communes].notna().mean()*100; rempl_test=test[cols_communes].notna().mean()*100
ecarts=pd.DataFrame({'train_rempli_%':rempl_train,'test_rempli_%':rempl_test})
ecarts['ecart_points']=(ecarts['train_rempli_%']-ecarts['test_rempli_%']).abs(); ecarts=ecarts.sort_values('ecart_points',ascending=False)
colonne_trouvee=str(ecarts.index[0]); ecart_remplissage=float(ecarts.iloc[0]['ecart_points'])
col2=str(ecarts.index[1]) if len(ecarts)>1 else colonne_trouvee
pourquoi_pas_la_derivee=f"La deuxième colonne {col2} a un écart de {float(ecarts.iloc[1]['ecart_points']) if len(ecarts)>1 else 0:.2f} points ; cela peut refléter une dérive de collecte ou de population, sans prouver une disparition complète."
remplissage_du_test=float(rempl_test.get(colonne_trouvee,0))
# Si la colonne est dans A_ECARTER, on la réintroduit temporairement pour mesurer son gain potentiel.
Xaug=X.copy()
if colonne_trouvee not in Xaug.columns:
    vals=train.loc[Xaug.index,colonne_trouvee]
    if not pd.api.types.is_numeric_dtype(vals):
        if vals.nunique(dropna=True)>255:
            vals=pd.Series(pd.factorize(vals.fillna('__MANQUANT__'))[0],index=vals.index,dtype='int64')
        else:
            vals=vals.fillna('').astype('category')
    Xaug[colonne_trouvee]=vals
Xtr_aug=Xaug.loc[Xtr.index]; Xval_aug=Xaug.loc[Xval.index]
m_avec=HistGradientBoostingClassifier(random_state=0).fit(Xtr_aug,ytr)
score_avec=100*m_avec.score(Xval_aug,yval)
Xtr_sans=Xtr_aug.drop(columns=[colonne_trouvee]); Xval_sans=Xval_aug.drop(columns=[colonne_trouvee])
m_sans=HistGradientBoostingClassifier(random_state=0).fit(Xtr_sans,ytr)
score_sans=100*m_sans.score(Xval_sans,yval)
gain_interne=score_avec-score_sans; gain_en_barres=gain_interne/max(float(erreur_type),1e-9)
# Rend manquante dans la validation la même proportion de lignes que dans le test.
Xsim=Xval_aug.copy(); proportion_trous=1-remplissage_du_test/100
ntrous=min(len(Xsim),int(round(proportion_trous*len(Xsim))))
rngsim=np.random.default_rng(123)
if ntrous: Xsim.iloc[rngsim.choice(len(Xsim),ntrous,replace=False),Xsim.columns.get_loc(colonne_trouvee)]=np.nan
score_simule=100*float(m_avec.score(Xsim,yval))
chute_en_barres=(score_simule-score_avec)/max(float(erreur_type),1e-9)
decision_A_ECARTER=f"L'écarter est prudent si cette colonne n'est pas disponible au moment de la prédiction : elle n'est renseignée qu'à {remplissage_du_test:.2f} % dans le test. Son gain interne est {gain_interne:+.2f} points ({gain_en_barres:+.2f} barre(s)); avec des trous simulés au taux du test, le score devient {score_simule:.2f} % ({chute_en_barres:+.2f} barre(s)). La décision repose sur la disponibilité en production, pas seulement sur le score interne."
display(ecarts.head(10))

### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
# Q12 : outil de recherche des signes VIDE et VALEUR

def suspects(df,cible):
    yloc=df[cible]; lignes=[]
    ids={ID,'id','id_client','patient_id','dossier_id','ligne_id'}
    for col in df.columns:
        if col==cible or col in ids or col.lower().endswith('_id'): continue
        s=df[col]; groups=list(pd.Series(yloc.dropna().unique()).tolist())
        if len(groups)<2: continue
        fills={g:float(s[yloc==g].notna().mean()) for g in groups}
        ecart=100*(max(fills.values())-min(fills.values()))
        if ecart>50:
            lignes.append({'colonne':col,'ecart_remplissage_points':ecart,'signe':'VIDE : remplissage très différent selon la cible','details':str({str(k):round(v*100,1) for k,v in fills.items()})})
        if s.nunique(dropna=True)<=100:
            ss=s.astype('object').where(s.notna(),'__MANQUANT__')
            exclusive=ss.groupby(yloc).apply(lambda z:set(z.dropna().unique()))
            if len(exclusive)>=2:
                inter=set.intersection(*[set(v) for v in exclusive.tolist()])
                union=set.union(*[set(v) for v in exclusive.tolist()])
                vals=[v for v in union-inter if v!='__MANQUANT__']
                if vals: lignes.append({'colonne':col,'ecart_remplissage_points':ecart,'signe':'VALEUR : modalité présente dans un seul groupe','details':f'modalités exclusives : {vals[:5]}'})
    return pd.DataFrame(lignes,columns=['colonne','ecart_remplissage_points','signe','details']).drop_duplicates()

table_suspects=suspects(train,CIBLE)
def classer_suspect(row):
    c=str(row['colonne']); sig=str(row['signe']).lower()
    if c in {'motif_absence','date_prise','date_rdv','duree_consultation','constantes_relevees'}:
        v='fuite'; j=f"{c} peut être connue après le rendez-vous ou dépendre de son issue ; son moment de disponibilité doit être vérifié."
    elif c.lower().endswith('_id') or c in {'ligne_id','patient_id','dossier_id'}:
        v='pas une fuite'; j=f"{c} identifie une ligne ou une personne, mais ne décrit pas directement le risque ; elle doit être exclue du modèle."
    elif 'vide' in sig:
        v='à discuter'; j=f"Pour {c}, le remplissage varie entre groupes ; cela peut signaler une collecte différente ou une fuite, à vérifier selon le sens métier et le temps."
    else:
        v='à discuter'; j=f"Pour {c}, une modalité n'apparaît que dans un groupe cible ; elle peut être rare ou liée à l'issue et sa signification doit être vérifiée."
    return pd.Series({'verdict':v,'justification':j})
if len(table_suspects): classement_suspects=table_suspects.apply(classer_suspect,axis=1)
else:
    table_suspects=pd.DataFrame([{'colonne':'aucune colonne détectée','ecart_remplissage_points':0.0,'signe':'aucun signe','details':'aucun suspect détecté'}])
    classement_suspects=pd.DataFrame([{'verdict':'pas une fuite','justification':'Aucun signe automatique détecté ; cela ne garantit pas l’absence de fuite et une vérification métier reste nécessaire.'}])
if 'age' in train.columns and (pd.to_numeric(train['age'],errors='coerce')==-1).any(): sent_col='age'; sent_val=-1
else:
    sent_col=None; sent_val=None
    for c in train.select_dtypes(include='number').columns:
        vals=train[c].value_counts(dropna=True); rare=vals[(vals.index<0)&(vals.values<500)]
        if len(rare): sent_col=c; sent_val=rare.index[0]; break
if sent_col is not None:
    mask=pd.to_numeric(train[sent_col],errors='coerce')==sent_val; lignes_porteuses=int(mask.sum())
    taux_sur_ces_lignes=100*float(train.loc[mask,CIBLE].mean()) if lignes_porteuses else 0.0; sentinelle_trouvee=f'{sent_col}={sent_val}'
    verdict_sentinelle=f"La valeur {sent_val} de {sent_col} apparaît sur {lignes_porteuses} lignes ; elle ressemble à un code de mesure manquante. Le taux d'absence est {taux_sur_ces_lignes:.2f} % contre {100*train[CIBLE].mean():.2f} % globalement : défaut de saisie à traiter, et fuite seulement si le code est attribué après l'issue."
else:
    sentinelle_trouvee='aucune sentinelle numérique rare identifiée'; lignes_porteuses=1; taux_sur_ces_lignes=100*float(train[CIBLE].mean()); verdict_sentinelle="Aucune sentinelle rare identifiée automatiquement ; consulter le dictionnaire des données avant de conclure qu'il n'en existe pas."
display(table_suspects); display(classement_suspects)

### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
# Q13 : annonce avant d'examiner les tranches
annonce="Je prévois que l'écart global entre rappel et absence ne suffit pas à démontrer une causalité. À délai égal, l'écart pourrait diminuer ou changer de signe ; je changerais d'avis si plusieurs tranches assez grandes montraient un effet de même sens dépassant deux barres."
col_rappel=next((c for c in mon_train.columns if 'rappel' in c.lower() or 'sms' in c.lower()),None)
col_prise=next((c for c in mon_train.columns if c.lower() in {'date_prise','appointment_registration','scheduledday'}),None)
col_rdv=next((c for c in mon_train.columns if c.lower() in {'date_rdv','appointmentday','appointment_day'}),None)
if col_rappel is not None:
    rappel=pd.to_numeric(mon_train[col_rappel],errors='coerce'); modalites=rappel.dropna().value_counts().index.tolist()[:2]
else: rappel=pd.Series(np.nan,index=mon_train.index); modalites=[]
avec=mon_train.loc[rappel==max(modalites),CIBLE] if modalites else pd.Series(dtype=float)
sans=mon_train.loc[rappel==min(modalites),CIBLE] if modalites else pd.Series(dtype=float)
taux_avec=100*float(avec.mean()) if len(avec) else 0.0; taux_sans=100*float(sans.mean()) if len(sans) else 0.0
ecart_global=taux_avec-taux_sans
if col_prise and col_rdv:
    delai=(pd.to_datetime(mon_train[col_rdv],errors='coerce',dayfirst=True)-pd.to_datetime(mon_train[col_prise],errors='coerce',dayfirst=True)).dt.total_seconds()/86400
else: delai=pd.Series(np.nan,index=mon_train.index)
tranches=[]; tranches_exclues=[]
if col_rappel is not None and delai.notna().any() and len(modalites)>=2:
    tmp=pd.DataFrame({'delai':delai,'rappel':rappel,'y':mon_train[CIBLE]}).dropna()
    tmp['tranche']=pd.qcut(tmp['delai'],q=5,duplicates='drop')
    for t,g in tmp.groupby('tranche',observed=True):
        ga=g[g.rappel==max(modalites)]; gs=g[g.rappel==min(modalites)]; n=len(ga)+len(gs)
        if not len(ga) or not len(gs) or n<500:
            tranches_exclues.append({'tranche':str(t),'lignes':n,'raison':'moins de 500 lignes ou un groupe absent'}); continue
        a=float(ga.y.mean()); b=float(gs.y.mean()); diff=100*(a-b); barre=100*np.sqrt(a*(1-a)/len(ga)+b*(1-b)/len(gs))
        tranches.append({'tranche':str(t),'taux_avec':100*a,'taux_sans':100*b,'n_avec':len(ga),'n_sans':len(gs),'lignes':n,'ecart':diff,'barre':barre,'ecart_en_barres':diff/max(barre,1e-9)})
    delai_median_avec=float(delai.loc[rappel==max(modalites)].median()); delai_median_sans=float(delai.loc[rappel==min(modalites)].median())
    variable_qui_explique=f"Délai médian : {delai_median_avec:.2f} jours dans le groupe de rappel élevé et {delai_median_sans:.2f} dans l'autre ; ce déséquilibre peut confondre l'écart global, sans prouver une causalité."
else:
    delai_median_avec=0.0; delai_median_sans=0.0; variable_qui_explique='Les colonnes de rappel et/ou de dates n’ont pas pu être identifiées avec certitude ; confirmer leur nom avant de conclure.'
experience_qui_trancherait="Tirer au hasard des rendez-vous éligibles et répartir aléatoirement les patients entre rappel et absence de rappel.\nComparer le taux d'absence entre groupes avec la même période et les mêmes règles de collecte.\nLa randomisation équilibre en moyenne les autres facteurs et rend la comparaison causale plus honnête."
display(pd.DataFrame(tranches)); display(pd.DataFrame(tranches_exclues))

### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
# Q14 : trois hypothèses et test de l'une d'elles
hypothese("indicateur_age_eleve","un âge élevé pourrait être associé à un taux d'absence plus faible ; signe attendu négatif","ajouter un indicateur age >= 60 et comparer le score sur la même validation","hypothèse contredite si le gain baisse de plus de deux barres ou si le taux augmente nettement chez les personnes âgées")
hypothese("delai_long_rdv","un délai plus long pourrait augmenter l'absence ; signe attendu positif","calculer le délai si les deux dates existent, puis comparer des tranches d'au moins 500 lignes","hypothèse contredite si l'écart est négatif et dépasse deux barres dans plusieurs tranches suffisamment grandes")
hypothese("nb_champs_numeriques_renseignes","une ligne plus complète pourrait être associée à moins d'absences ; signe attendu négatif","comparer le score de base et celui avec le nombre de champs numériques renseignés","hypothèse contredite si le gain est inférieur à moins deux barres sur la validation fixée")
nom_test='indicateur_age_eleve'; prevoir(nom_test,'gain faible ou nul plausible','le modèle peut déjà capter l’information liée à l’âge')
def mesure_age():
    if 'age' not in X.columns: return modele.score(Xval,yval)
    a=Xtr.copy(); b=Xval.copy(); a[nom_test]=(pd.to_numeric(a['age'],errors='coerce')>=60).astype(int); b[nom_test]=(pd.to_numeric(b['age'],errors='coerce')>=60).astype(int)
    m=HistGradientBoostingClassifier(random_state=0).fit(a,ytr); return m.score(b,yval)
score_hypothese=essai(nom_test,mesure_age,'indicateur age >= 60','gain faible ou nul plausible')
score_hypothese_pct=score_interne if score_hypothese is None else score_hypothese
gain_hypothese_barres=(score_hypothese_pct-score_interne)/max(float(erreur_type),1e-9)
confirmee_ou_contredite='confirmée' if gain_hypothese_barres>=-2 else 'contredite'
suite_donnee=f"Le gain mesuré est de {gain_hypothese_barres:+.2f} barre(s) ; l'hypothèse est {confirmee_ou_contredite} selon le critère annoncé. Je garderais ce résultat comme exploratoire et vérifierais le signe sur des groupes assez grands et un second découpage avant toute décision de production."

### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
# Q15 : classification et justifications des colonnes de crédit
familles={"id_client":"identifiant","date_defaut":"connue après l'issue","statut_6_mois":"connue après l'issue","montant":"utilisable","canal":"risque de disparition","score_interne":"utilisable"}
justifications={
"id_client":"Cette colonne désigne le client ; elle sert à l'identification et ne décrit pas directement le risque de la demande.",
"date_defaut":"La date du défaut n'est renseignée qu'après l'événement et n'existe donc pas au moment d'accorder le crédit.",
"statut_6_mois":"Ce statut est relevé six mois après l'octroi ; il s'agit d'une information future par rapport à la décision.",
"montant":"Le montant demandé est connu au moment de la demande et décrit directement le crédit à évaluer.",
"canal":"Le canal est connu au moment de la demande, mais le système de collecte peut changer ; sa disponibilité durable doit être vérifiée.",
"score_interne":"Ce score est calculé au moment de la demande ; il est disponible à la décision et décrit le risque évalué."}

## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
